In [1]:
import pandas as pd
import datetime

In [2]:
df = pd.read_excel("/content/att feb 2026.xls", engine="xlrd", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Emp Code : 10005,NaN,NaN,Emp Name : Gayathri,NaN,NaN,NaN,NaN,Present : 20.00,NaN,...,Leave : 0.00,NaN,OT Hrs : 0:00,NaN,Work Hrs : 173:07,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
4,NaN,Su,Mo,Tu,We,Th,Fr,Sa,Su,Mo,...,Th,Fr,Sa,Su,Mo,Tu,We,Th,Fr,Sa


In [3]:
def calc_work_hours(in_t, out_t):
    if not in_t or not out_t or ":" not in str(in_t) or ":" not in str(out_t):
        return "0:00"
    try:
        # Updated to handle HH:MM:SS format found in the Excel
        fmt = "%H:%M:%S" if str(in_t).count(":") == 2 else "%H:%M"
        t1 = datetime.datetime.strptime(str(in_t), fmt)
        t2 = datetime.datetime.strptime(str(out_t), fmt)
        diff = t2 - t1
        sec = diff.total_seconds()
        if sec < 0: sec += 86400  # Handle overnight shifts
        return f"{int(sec//3600)}:{int((sec%3600)//60):02d}"
    except Exception as e:
        return "0:00"

In [4]:
records = []
sno = 1
year = int(input("Enter year: "))
month = int(input("Enter month: "))
print(f"Year set to {year}, Month set to {month}")

Enter year: 2026
Enter month: 2
Year set to 2026, Month set to 2


In [5]:
records = []
current_emp_id = None
current_emp_name = None

for idx, row in df.iterrows():
    row_str = " ".join(row.astype(str))

    # Capture Employee Details
    if "Emp Code :" in row_str:
        for val in row:
            text = str(val)
            if "Emp Code :" in text: current_emp_id = text.split(":")[1].strip()
            if "Emp Name :" in text: current_emp_name = text.split(":")[1].strip()
        continue

    # Identify the row containing attendance data
    if isinstance(row.iloc[0], str) and "Shift" in row.iloc[0]:
        # Columns 1 to 28/29/30/31 contain the data for each day
        for day_num in range(1, len(df.columns)):
            cell_val = row.iloc[day_num]
            if pd.notna(cell_val) and "\n" in str(cell_val):
                parts = str(cell_val).split("\n")
                if len(parts) >= 8:
                    try:
                        # Create date string
                        date_obj = datetime.datetime(year, month, day_num)
                        in_t = parts[1].strip()
                        out_t = parts[2].strip()

                        records.append({
                            "sno": sno,
                            "emp_id": current_emp_id,
                            "emp_name": current_emp_name,
                            "date": date_obj.strftime("%Y-%m-%d"),
                            "day_name": date_obj.strftime("%A"),
                            "in_time": in_t,
                            "out_time": out_t,
                            "work_hours": parts[6].strip(),
                            "calc_hrs": calc_work_hours(in_t, out_t),
                            "status": parts[7].strip()
                        })
                        sno += 1
                    except Exception:
                        continue

attendance_df = pd.DataFrame(records)
print(f"Extracted {len(attendance_df)} records.")
attendance_df.head()

Extracted 224 records.


,sno,emp_id,emp_name,date,day_name,in_time,out_time,work_hours,calc_hrs,status
0,1,10005,Gayathri,2026-02-01,Sunday,,,0:00,0:00,WO-I
1,2,10005,Gayathri,2026-02-02,Monday,09:41,18:44,9:03,9:03,P
2,3,10005,Gayathri,2026-02-03,Tuesday,09:38,18:42,9:04,9:04,P
3,4,10005,Gayathri,2026-02-04,Wednesday,09:54,18:30,8:36,8:36,P
4,5,10005,Gayathri,2026-02-05,Thursday,10:06,19:06,9:00,9:00,P


In [6]:
def hhmm_to_minutes(hhmm):
    if not isinstance(hhmm, str) or ":" not in hhmm:
        return 0
    h, m = hhmm.split(":")
    return int(h) * 60 + int(m)

def normalize_work_hours_real(actual_minutes,
                              unit=60,
                              grace=5,
                              min_hours=1,
                              max_hours=10):
    """
    Industry-style normalization using bucket + grace.
    """
    # Safety cap (no overtime)
    actual_minutes = min(actual_minutes, max_hours * unit)

    base_bucket = (actual_minutes // unit) * unit
    next_bucket = base_bucket + unit

    # Distance to next bucket
    if next_bucket - actual_minutes <= grace:
        normalized_minutes = next_bucket
    else:
        normalized_minutes = base_bucket

    normalized_hours = normalized_minutes // unit

    # Enforce minimum payable
    if normalized_hours < min_hours:
        return 0

    return min(normalized_hours, max_hours)

def get_normalized_hours(hhmm_str):
    mins = hhmm_to_minutes(hhmm_str)
    return normalize_work_hours_real(mins)

In [16]:
attendance_df['normalized_work_hrs'] = attendance_df['calc_hrs'].apply(get_normalized_hours)
display(attendance_df[['emp_id', 'emp_name','date', 'day_name','in_time', 'out_time', 'calc_hrs', 'normalized_work_hrs']].head(20))

,emp_id,emp_name,date,day_name,in_time,out_time,calc_hrs,normalized_work_hrs
0,10005,Gayathri,2026-02-01,Sunday,,,0:00,0
1,10005,Gayathri,2026-02-02,Monday,09:41,18:44,9:03,9
2,10005,Gayathri,2026-02-03,Tuesday,09:38,18:42,9:04,9
3,10005,Gayathri,2026-02-04,Wednesday,09:54,18:30,8:36,8
4,10005,Gayathri,2026-02-05,Thursday,10:06,19:06,9:00,9
5,10005,Gayathri,2026-02-06,Friday,,,0:00,0
6,10005,Gayathri,2026-02-07,Saturday,09:44,18:51,9:07,9
7,10005,Gayathri,2026-02-08,Sunday,,,0:00,0
8,10005,Gayathri,2026-02-09,Monday,09:57,18:58,9:01,9
9,10005,Gayathri,2026-02-10,Tuesday,09:59,18:59,9:00,9


In [17]:
attendance_df.to_excel('Biometeric_attendance_cleaned.xlsx', index=False)
print('Biometeric_attendance_cleaned.excel is download.')

Biometeric_attendance_cleaned.excel is download.


extract and normalize done

In [24]:
spd=pd.read_excel("/content/Biometeric_attendance_cleaned.xlsx")

In [25]:
salarypd=pd.read_excel("/content/Bio_Salary.xlsx")

In [26]:
grouped_df = (
    spd.groupby("emp_id", as_index=False)
       .agg(
           total_worked_hours=("normalized_work_hrs", "sum"),
           daily_worked_list=("normalized_work_hrs", list)
       )
)

In [27]:
payroll_df = grouped_df.merge(
    salarypd,
    on="emp_id",
    how="left"
)

In [22]:
TOTAL_DAYS =int(input("Enter the number of days:--"))

Enter the number of days:--28


In [23]:
WEEKOFF_AND_PAID_LEAVE =int(input("Enter the number of paid leave:--"))

Enter the number of paid leave:--5


In [28]:
HOURS_PER_DAY= payroll_df["daily_working_hours"]
Total_hours = TOTAL_DAYS * HOURS_PER_DAY

In [29]:
payroll_df["total_days"] = TOTAL_DAYS
payroll_df["weekoff_paid_leave"] = WEEKOFF_AND_PAID_LEAVE

In [30]:
payroll_df["total_expected_hours"] = (    (TOTAL_DAYS - WEEKOFF_AND_PAID_LEAVE)    * payroll_df["daily_working_hours"])

In [31]:
payroll_df.loc[payroll_df["total_expected_hours"] < 0, "total_expected_hours"] = 0

In [32]:
payroll_df.columns = payroll_df.columns.str.strip()

payroll_df["salary_hour"] = payroll_df["salary"] / Total_hours

print("Calculated daily_working_hours successfully.")

Calculated daily_working_hours successfully.


In [33]:
payroll_df.loc[payroll_df["total_expected_hours"] == 0, "daily_working_hours"] = 0

In [34]:
payroll_df["base_salary"] = (payroll_df["total_worked_hours"] * payroll_df["salary_hour"])

In [35]:
payroll_df["Paid_Weekend"] = WEEKOFF_AND_PAID_LEAVE * (payroll_df["salary_hour"] * payroll_df["daily_working_hours"])

In [36]:
def calculate_leverage(row):

    daily_list = row["daily_worked_list"]
    assigned_hours = row["daily_working_hours"]

    # 1. Take only partial days (< assigned and not 0)
    eligible = [h for h in daily_list if 0 < h < assigned_hours]

    if not eligible:
        return pd.Series([0, 0])

    # 2. Sort ascending (worst days first)
    eligible.sort()

    # 3. Take max 3 days
    selected = eligible[:3]

    n = len(selected)

    # 4. Your exact formula
    difference = (n * assigned_hours) - sum(selected)

    # 5. Cap at 3 hours max
    leverage_hours = min(max(difference, 0), 3)

    leverage_amount = leverage_hours * row["salary_hour"]

    return pd.Series([leverage_hours, leverage_amount])


payroll_df[["leverage_hr", "leverage_amount"]] = payroll_df.apply(
    calculate_leverage,
    axis=1
)

In [37]:
payroll_df["net_salary"] = payroll_df["base_salary"] + payroll_df["Paid_Weekend"] + payroll_df["leverage_amount"]

In [38]:
import numpy as np

# Ensure columns are stripped of spaces
payroll_df.columns = payroll_df.columns.str.strip()
payroll_df["deduction"] = np.select(
    [
        payroll_df["salary"] <= 21000,
        (payroll_df["salary"] >= 21001) & (payroll_df["salary"] <= 24999),
        payroll_df["salary"] >= 25000
    ],
    [
        (payroll_df["net_salary"] * 0.0075),  # ESI 0.75%
        0,                                 # No deduction
        200                                # Fixed PD
    ],
    default=0
)

In [39]:
payroll_df["Final_salary"] =( payroll_df["net_salary"] - payroll_df["deduction"].round(2))
display(payroll_df.head())

,emp_id,total_worked_hours,daily_worked_list,daily_working_hours,salary,total_days,weekoff_paid_leave,total_expected_hours,salary_hour,base_salary,Paid_Weekend,leverage_hr,leverage_amount,net_salary,deduction,Final_salary
0,10005,171,"[0, 9, 9, 8, 9, 0, 9, 0, 9, 9, 9, 9, 9, 9, 0, ...",9,20000,28,5,207,79.365079,13571.428571,3571.428571,3.0,238.095238,17380.952381,130.357143,17250.592381
1,10006,207,"[0, 9, 9, 9, 9, 9, 9, 0, 9, 9, 9, 9, 0, 9, 0, ...",9,15000,28,5,207,59.523810,12321.428571,2678.571429,0.0,0.000000,15000.000000,112.500000,14887.500000
2,10008,189,"[0, 9, 9, 9, 9, 9, 9, 0, 0, 9, 9, 9, 0, 9, 0, ...",9,20000,28,5,207,79.365079,15000.000000,3571.428571,0.0,0.000000,18571.428571,139.285714,18432.138571
3,10009,183,"[0, 7, 9, 9, 9, 9, 9, 0, 9, 9, 9, 7, 0, 0, 0, ...",10,25000,28,5,230,89.285714,16339.285714,4464.285714,3.0,267.857143,21071.428571,200.000000,20871.428571
4,10011,123,"[0, 9, 9, 8, 9, 9, 8, 0, 0, 9, 9, 9, 9, 0, 0, ...",10,25000,28,5,230,89.285714,10982.142857,4464.285714,3.0,267.857143,15714.285714,200.000000,15514.285714


In [41]:
# Define the desired column order
column_order = [
    'emp_id',
    'salary',
    'total_days',
    'weekoff_paid_leave',
    'daily_working_hours',
    'total_expected_hours',
    'daily_worked_list',
    'total_worked_hours',
    'salary_hour',
    'base_salary',
    'Paid_Weekend',
    'leverage_hr',
    'leverage_amount',
    'deduction',
    'net_salary',
    'Final_salary'

]

# Filter the dataframe to only include these columns in this specific order
# We check if columns exist to avoid errors
existing_cols = [col for col in column_order if col in payroll_df.columns]
payroll_report = payroll_df[existing_cols].copy()

# Display the rearranged dataframe
display(payroll_report.head())
payroll_report.to_excel("payroll_report.xlsx", index=False)

,emp_id,salary,total_days,weekoff_paid_leave,daily_working_hours,total_expected_hours,daily_worked_list,total_worked_hours,salary_hour,base_salary,Paid_Weekend,leverage_hr,leverage_amount,deduction,net_salary,Final_salary
0,10005,20000,28,5,9,207,"[0, 9, 9, 8, 9, 0, 9, 0, 9, 9, 9, 9, 9, 9, 0, ...",171,79.365079,13571.428571,3571.428571,3.0,238.095238,130.357143,17380.952381,17250.592381
1,10006,15000,28,5,9,207,"[0, 9, 9, 9, 9, 9, 9, 0, 9, 9, 9, 9, 0, 9, 0, ...",207,59.523810,12321.428571,2678.571429,0.0,0.000000,112.500000,15000.000000,14887.500000
2,10008,20000,28,5,9,207,"[0, 9, 9, 9, 9, 9, 9, 0, 0, 9, 9, 9, 0, 9, 0, ...",189,79.365079,15000.000000,3571.428571,0.0,0.000000,139.285714,18571.428571,18432.138571
3,10009,25000,28,5,10,230,"[0, 7, 9, 9, 9, 9, 9, 0, 9, 9, 9, 7, 0, 0, 0, ...",183,89.285714,16339.285714,4464.285714,3.0,267.857143,200.000000,21071.428571,20871.428571
4,10011,25000,28,5,10,230,"[0, 9, 9, 8, 9, 9, 8, 0, 0, 9, 9, 9, 9, 0, 0, ...",123,89.285714,10982.142857,4464.285714,3.0,267.857143,200.000000,15714.285714,15514.285714
